In [ ]:
import trafilatura as trf
import getpass
from collections.abc import Sequence, Mapping
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Setup:
OPENAI_API_KEY = getpass.getpass('Enter your open AI API key:')
llm = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    model_name='gpt-5-nano',
)

In [ ]:
def extract(url: str) -> str:
    return trf.extract(trf.fetch_url(url))    

In [ ]:
list_of_urls = [
    'https://www.nationalgallery.org.uk/paintings/learn-about-art/guide-to-impressionism',
    'https://www.tate.org.uk/art/art-terms/i/impressionism',
    'https://www.artic.edu/highlights/5/impressionism',
    'https://en.wikipedia.org/wiki/Impressionism',
]

texts = [extract(url) for url in list_of_urls]

In [ ]:
doc_summary_template = '''
Write a concise summary of the following text:
{text}
DOC SUMMARY:
'''
doc_summary_prompt = PromptTemplate.from_template(doc_summary_template)
doc_summary_chain = doc_summary_prompt | llm | StrOutputParser()


refine_summary_template = '''
Your must produce a final summary from the current refined summary
which has been generated so far and from the content of an additional document.
This is the current refined summary generated so far:
{current_refined_summary}

This is the content of the additional document:
{text}

Only use the content of the additional document if it is useful, 
otherwise return the current full summary as it is.
'''

refine_summary_prompt = PromptTemplate.from_template(refine_summary_template)
refine_chain = refine_summary_prompt | llm | StrOutputParser()

In [ ]:
def refine_summary(docs):
    intermediate_steps = []
    current_refined_summary = doc_summary_chain.invoke({
        'text': docs[0],
    })
        
    for doc in docs[1:]:
        intermediate_step = {
            'current_refined_summary': current_refined_summary, 
            'text': doc
        }
        intermediate_steps.append(intermediate_step)        
        current_refined_summary = refine_chain.invoke(intermediate_step)
        
    return {
        'final_summary': current_refined_summary,
        'intermediate_steps': intermediate_steps
    }

In [ ]:
full_summary = refine_summary(texts)

In [ ]:
print(full_summary['intermediate_steps'][0]['current_refined_summary'])

In [ ]:
print(full_summary['intermediate_steps'][1]['current_refined_summary'])

In [ ]:
print(full_summary['intermediate_steps'][2]['current_refined_summary'])

In [ ]:
print(full_summary['final_summary'])